In [1]:
puts `ls -l raw-data`

total 9556
-rw-rw-r-- 1 osboxes osboxes   34777 Apr 15 13:32 Demokritos-KG-information.xlsx
-rw-rw-r-- 1 osboxes osboxes  402701 Apr 15 13:32 Disease-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes  125055 Apr 15 13:32 Disease-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes   20901 Apr 15 13:32 Disease-Gene triples.tsv
-rw-rw-r-- 1 osboxes osboxes       0 Apr 15 16:21 disease_list.txt
-rw-rw-r-- 1 osboxes osboxes  207331 Apr 15 13:32 Disease-Therapeutic_Area.tsv
-rw-rw-r-- 1 osboxes osboxes   82213 Apr 15 13:32 Drug-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes 7769112 Apr 15 13:32 Drug-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  111643 Apr 15 13:32 Drug-Drug_type.tsv
-rw-rw-r-- 1 osboxes osboxes   87477 Apr 15 13:32 Drug-Gene triples.tsv
drwxrwxr-x 2 osboxes osboxes    4096 Apr  7 15:55 Feb 2026
-rw-rw-r-- 1 osboxes osboxes  684226 Apr 15 13:32 Gene-Disease triples.tsv
-rw-rw-r-- 1 osboxes osboxes   57380 Apr 15 13:32 Gene-Drug triples.tsv
-rw-rw-r-- 1 osboxes osboxes  127519 Apr

Disease-Drug triples.tsv COLUMN 6
Gene-Drug triples.tsv COLUMN 6
Drug-Disease triples.tsv COLUMN 2
Drug-Drug triples.tsv  COLUMN 2 and 6
Drug-Gene triples.tsv  COLUMN 2

In [22]:
puts `awk -F'\t' '{print $6}' "raw-data/Disease-Drug triples.tsv" | sort | uniq > drug_list.txt`
puts `cat drug_list.txt | wc -l`


558


In [23]:
puts `awk -F'\t' '{print $6}' "raw-data/Gene-Drug triples.tsv" | sort | uniq >> drug_list.txt`
puts `cat drug_list.txt | wc -l`


880


In [24]:
puts `awk -F'\t' '{print $2}' "raw-data/Drug-Disease triples.tsv" | sort | uniq >> drug_list.txt`
puts `cat drug_list.txt | wc -l`


1305


In [6]:
# This file contains errors in column 6
# puts `awk -F'\t' '{print $6}' "raw-data/Drug-Drug triples.tsv" | sort | uniq >> drug_list.txt`
# puts `cat drug_list.txt | wc -l`


29368


In [5]:
# This file contains errors in column 6
# puts `awk -F'\t' '{print $2}' "raw-data/Drug-Drug triples.tsv" | sort | uniq >> drug_list.txt`
# puts `cat drug_list.txt | wc -l`


7922


In [25]:
puts `awk -F'\t' '{print $2}' "raw-data/Drug-Gene triples.tsv" | sort | uniq >> drug_list.txt`
puts `cat drug_list.txt | wc -l`


1836


In [26]:
puts `(echo "demokritos_drug_cui"; cat drug_list.txt | sort | uniq) > drug_list_uniq.txt`
puts `head -5 drug_list_uniq.txt`
puts `cat drug_list_uniq.txt | wc -l`  # how many records in total?


demokritos_drug_cui
C0000376
C0000379
C0000407
C0000477
1214


In [28]:
cuis = File.read('drug_list_uniq.txt').split(/\n/)
puts cuis[-1]
puts cuis[-2]
puts cuis[-3]
puts cuis[-4]
#  Not sure why some are C-number ands others are just numbers...??

puts cuis[1..-2].last
cuis = cuis[1..-2]  # skip the final two, which are column headers from the original .tsv

Drug_id
C5544328
C5139806
C4520812
C5544328


["C0000376", "C0000379", "C0000407", "C0000477", "C0000578", "C0000610", "C0000970", "C0000981", "C0001002", "C0001041", "C0001046", "C0001047", "C0001443", "C0001617", "C0001641", "C0001644", "C0001645", "C0001648", "C0001771", "C0001888", "C0001927", "C0001933", "C0001962", "C0001963", "C0001992", "C0001994", "C0002006", "C0002062", "C0002078", "C0002144", "C0002151", "C0002333", "C0002348", "C0002403", "C0002475", "C0002482", "C0002502", "C0002508", "C0002520", "C0002525", "C0002598", "C0002607", "C0002615", "C0002658", "C0002679", "C0002712", "C0002763", "C0002771", "C0002844", "C0002932", "C0002934", "C0003009", "C0003015", "C0003195", "C0003209", "C0003211", "C0003286", "C0003289", "C0003295", "C0003297", "C0003299", "C0003360", "C0003364", "C0003372", "C0003385", "C0003402", "C0003417", "C0003438", "C0003440", "C0003596", "C0003620", "C0003765", "C0003968", "C0003999", "C0004057", "C0004147", "C0004234", "C0004259", "C0004480", "C0004609", "C0004906", "C0005014", "C0005064", "C0

# Drug_id is a UMLS term



# Map UMLS to PubChem CUI and formal name

In [29]:
require 'rest-client'
require 'json'

def map_umls_to_cid(cui)
  api_key = ENV["APIKEY"] # Replace with BioPortal API key
  begin
    # cui = "C0019247"
    url = "https://data.bioontology.org/search?q=#{cui}&ontologies=MESH,SNOMEDCT&require_exact_match=true&apikey=#{api_key}"
    warn url
    response = RestClient.get(url)
  rescue
    warn "umls lookup failed #{response.inspect}"
    return false
  end
  mappings = []
  data = JSON.parse(response)

#  if hit = data.dig('collection', 0)
  if hit = data.dig('collection')
    unless hit.first   # collection can be []
      warn "No data found for #{cui}\n"
      return false
    end
      
    hit.each do |h|
      compound_name = h&.dig('prefLabel')
      xref = h&.dig('@id')
      linksurl = h&.dig('links', 'mappings')
      mappings << { cui: cui, xref: xref, compound_name: compound_name, linksurl: linksurl}
    end
    return mappings
  else
    warn "No data found for #{cui}\n"
    return false
  end

rescue StandardError => e
  warn "No data found for #{cui} Error: #{e.inspect}\n"
  return false
end

# Example
puts map_umls_to_cid('C0042291')
  

https://data.bioontology.org/search?q=C0042291&ontologies=MESH,SNOMEDCT&require_exact_match=true&apikey=74027bd8-6be0-4329-be22-aa3717f97243


{:cui=>"C0042291", :xref=>"http://purl.bioontology.org/ontology/SNOMEDCT/387080000", :compound_name=>"Valproic acid", :linksurl=>"https://data.bioontology.org/ontologies/SNOMEDCT/classes/http%3A%2F%2Fpurl.bioontology.org%2Fontology%2FSNOMEDCT%2F387080000/mappings"}
{:cui=>"C0042291", :xref=>"http://purl.bioontology.org/ontology/SNOMEDCT/13965000", :compound_name=>"Valproic acid-containing product", :linksurl=>"https://data.bioontology.org/ontologies/SNOMEDCT/classes/http%3A%2F%2Fpurl.bioontology.org%2Fontology%2FSNOMEDCT%2F13965000/mappings"}


In [30]:
require 'rest-client'
require 'json'

def map_name_to_cid(name)
  # PubChem REST is stupid, and consumes names that are only partially URI encoded!  (spaces substituted) and rejects fully URI encoded strings!
  # so I a forced to roll my own URI escaper... so stupid!
  escname = name.gsub(/\s/, "%20")
  url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/#{escname}/cids/JSON"
  warn url
  begin
    response = RestClient.get(url)
  rescue
    warn "name #{name} lookup failed #{response.inspect}"
    return false
  end
  data = JSON.parse(response)
#   warn JSON.pretty_generate data
#   abort

  cids = JSON.parse(response.body).dig('IdentifierList', 'CID')
  
  { name: name, cid: cids&.first || 'No CID found' }
  
end

# Example
mappings = map_umls_to_cid('C0042291')
mappings.each do |map|
  puts map_name_to_cid(map[:compound_name])
end
puts



https://data.bioontology.org/search?q=C0042291&ontologies=MESH,SNOMEDCT&require_exact_match=true&apikey=74027bd8-6be0-4329-be22-aa3717f97243
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Valproic%20acid/cids/JSON


{:name=>"Valproic acid", :cid=>3121}


https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Valproic%20acid-containing%20product/cids/JSON
name Valproic acid-containing product lookup failed nil


false



# Iteration over all UMLS terms

In [31]:
require 'json'
require 'csv'

OUTPUT = "./maps/2026-drug-mappings.map".freeze
ERRORFILE = "./maps/2026-drug-mapping-errors.txt".freeze

error = File.open(ERRORFILE, "w")
error.sync = true
out = File.open(OUTPUT, "w")
out.sync = true
out.write CSV.generate_line(["demokritosid", "xref", "pubchem_cid", "IUPACname"])


cuis.each do |cui|
  
  # first lookup cui
  mappings = map_umls_to_cid(cui) 
  # {:cui=>"C0613621", :mesh=>"http://purl.bioontology.org/ontology/MESH/C030536", :compound_name=>"2,2-dichloro-1,1-difluoroethyl difluoromethyl ether", :linksurl=>"https://data.bioontology.org/ontologies/MESH/classes/http%3A%2F%2Fpurl.bioontology.org%2Fontology%2FMESH%2FC030536/mappings"}
  unless mappings
    warn "failed UMLS to cid lookup for #{cui}"
    error.write "failed UMLS to cid lookup for #{cui}\n"
    next
  end
  
  result = []
  mappings.each do |map|
    xref = map[:xref]
    hash = map_name_to_cid(map[:compound_name]) # {:name=>"2,2-dichloro-1,1-difluoroethyl%20difluoromethyl%20ether", :cid=>152803}
    unless hash
      warn "failed compound name to cid lookup for XREF #{cui} #{map[:compound_name]}"
      error.write "failed compound name to cid lookup for XREF #{cui} #{map[:compound_name]}\n"
      next
    end
    cid = hash[:cid]
    cid_guid = "https://pubchem.ncbi.nlm.nih.gov/compound/#{cid}"
    iupacname = hash[:name]
    out.write CSV.generate_line([cui, xref, cid_guid, iupacname])
  end
end

puts "DONE!"
  
  
  

(irb):3: warning: already initialized constant Object::OUTPUT
(irb):3: warning: previous definition of OUTPUT was here
(irb):4: warning: already initialized constant Object::ERRORFILE
(irb):4: warning: previous definition of ERRORFILE was here
https://data.bioontology.org/search?q=C0000376&ontologies=MESH,SNOMEDCT&require_exact_match=true&apikey=74027bd8-6be0-4329-be22-aa3717f97243
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/3,4-Dihydroxyphenylacetic%20Acid/cids/JSON
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/Dihydroxyphenylacetic%20acid/cids/JSON
name Dihydroxyphenylacetic acid lookup failed nil
failed compound name to cid lookup for XREF C0000376 Dihydroxyphenylacetic acid
https://data.bioontology.org/search?q=C0000379&ontologies=MESH,SNOMEDCT&require_exact_match=true&apikey=74027bd8-6be0-4329-be22-aa3717f97243
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/3,4-Methylenedioxyamphetamine/cids/JSON
https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/nam

DONE!


In [21]:
puts `cat #{OUTPUT} | wc -l`
puts `cat #{ERRORFILE} | wc -l`


4218
26031
